# Part 1: EDA

We began by examining the raw light curves directly to understand the basic structure of the data. The raw plots showed that flux magnitude varied dramatically across stars, with some series exhibiting large drifts, oscillations, or spikes that made direct comparison difficult. This confirmed that raw flux values were not on a common scale and that some stars contained substantial noise or artifact-like behavior.

To make stars comparable, we applied per-star z-score normalization and replotted the same examples. After normalization, differences in absolute scale were removed, making relative shape and dimming behavior much easier to compare. This step was important because our downstream engineered features were meant to capture *relative* transit-like behavior rather than raw brightness.

We also inspected the positive class more closely by plotting all exoplanet-labeled stars after normalization. This showed that the positive class was not uniform: some stars displayed repeated sharp dips, others showed smoother low-flux behavior, and some still contained spikes or irregularities. This suggested that confirmed exoplanet signals in the dataset are heterogeneous, which helps explain why class separation is only partial rather than clean.

Next, we created engineered feature distribution plots for several transit-motivated features. These showed that exoplanet-labeled stars tended to have:
- more negative low-tail behavior (`p01_z`, `min_z`),
- more time spent in unusually low flux (`frac_below_-2`),
- longer continuous low-flux runs (`longest_run_-1`),
- and lower local roughness (`mean_abs_diff`) than many negative examples.
These plots provided the first strong evidence that simple interpretable features could separate the classes.

Because some features were strongly skewed, we also used a skew-adjusted plot of `log1p(longest_run_-1)`. This made the run-length difference easier to see than the raw boxplot and confirmed that exoplanet stars generally had longer sustained low-flux episodes.

We then highlighted one strong single feature, `p01_z`, with a strip-over-box plot. This showed that the positive class generally had a deeper negative tail than the negative class, but with substantial overlap. This was useful for showing that individual features carry signal, while also making it clear that no single feature perfectly solves the task.

To quantify separation more directly, we computed feature effect size proxies comparing positives and negatives. This showed that the strongest engineered signals included:
- `frac_below_-2` and `longest_run_-1` (both larger for positives),
- `p01_z` (more negative for positives),
- and `mean_abs_diff` (smaller for positives).
This supported the interpretation that exoplanet stars often combine deeper, more sustained dimming with smoother local structure.

We also examined pairwise feature correlations to understand redundancy. The correlation heatmap showed clear groups of related features (for example, depth features, threshold-based dip-frequency features, and variability features), indicating that several engineered features were measuring similar aspects of the signal. This later motivated feature pruning and regularization in the inferential model.

A PCA projection of the engineered feature space showed that exoplanet and non-exoplanet stars were not cleanly separable. The positive class occupied a somewhat shifted region of the space, but there was still heavy overlap with the negative class. This reinforced that the task is nontrivial and justified the need for model-based classification rather than simple threshold rules alone.

We also checked robustness to preprocessing by comparing effect size behavior under z-score and robust (median/IQR) scaling. The key features kept the same directional trends under both methods, indicating that the main feature story was stable and not an artifact of one normalization choice.

Finally, we compared several important engineered features between train and test. The train/test histograms for `p01_z`, `frac_below_-2`, and `mean_abs_diff` were broadly similar, with no obvious large distribution shift. This supported the use of the provided train/test split and suggested that the engineered feature distributions were reasonably consistent across both sets.

## Main EDA takeaway

Overall, the EDA showed that the problem is difficult because the positive class is extremely small and overlaps heavily with the negative class, but also that exoplanet-labeled stars do display measurable differences in low-flux depth, low-flux duration, dip frequency, and local smoothness. These findings justified both the engineered-feature approach and the later use of regularized logistic regression and calibrated classification.

In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

TRAIN_PATH = "exoTrain.csv"
TEST_PATH  = "exoTest.csv"
os.makedirs("plots", exist_ok=True)

def load_exo(path: str):
    df = pd.read_csv(path)
    y_raw = df["LABEL"].astype(int).to_numpy()
    y = (y_raw == 2).astype(int)  # 1=exoplanet, 0=non-exoplanet
    flux_cols = [c for c in df.columns if c.startswith("FLUX.")]
    X = df[flux_cols].to_numpy(dtype=np.float32)
    return X, y, y_raw, flux_cols

X_train, y_train, yraw_train, flux_cols = load_exo(TRAIN_PATH)
X_test,  y_test,  yraw_test,  _         = load_exo(TEST_PATH)

def per_series_zscore(X: np.ndarray, eps: float = 1e-8):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    return (X - mu) / (sd + eps)

def per_series_robust(X: np.ndarray, eps: float = 1e-8):
    med = np.median(X, axis=1, keepdims=True)
    q75 = np.quantile(X, 0.75, axis=1, keepdims=True)
    q25 = np.quantile(X, 0.25, axis=1, keepdims=True)
    iqr = q75 - q25
    return (X - med) / (iqr + eps)

In [39]:
rng = np.random.default_rng(0)

def pick_indices(y, n_neg=8, n_pos=8):
    neg_idx = rng.choice(np.where(y==0)[0], size=n_neg, replace=False)
    pos_pool = np.where(y==1)[0]
    pos_idx = rng.choice(pos_pool, size=min(n_pos, len(pos_pool)), replace=False)
    return neg_idx, pos_idx

neg_idx, pos_idx = pick_indices(y_train, n_neg=8, n_pos=8)

Z_train = per_series_zscore(X_train)

def plot_lightcurves_panel(X_raw, X_norm, neg_idx, pos_idx, title_prefix="Train"):
    t = np.arange(X_raw.shape[1])

    fig, axes = plt.subplots(2, 2, figsize=(12,6), sharex=True)
    axes = axes.ravel()

    # raw negatives
    for i in neg_idx:
        axes[0].plot(t, X_raw[i], linewidth=1)
    axes[0].set_title(f"{title_prefix}: Non-exoplanet (raw)")

    # raw positives
    for i in pos_idx:
        axes[1].plot(t, X_raw[i], linewidth=1)
    axes[1].set_title(f"{title_prefix}: Exoplanet (raw)")

    # normalized negatives
    for i in neg_idx:
        axes[2].plot(t, X_norm[i], linewidth=1)
    axes[2].set_title(f"{title_prefix}: Non-exoplanet (z-score)")

    # normalized positives
    for i in pos_idx:
        axes[3].plot(t, X_norm[i], linewidth=1)
    axes[3].set_title(f"{title_prefix}: Exoplanet (z-score)")

    for ax in axes:
        ax.set_xlabel("time index")
        ax.set_ylabel("flux")
    plt.tight_layout()
    plt.savefig(f"./plots/lightcurves_panel.png")
    plt.close()

plot_lightcurves_panel(X_train, Z_train, neg_idx, pos_idx, "Train")

In [40]:
def longest_run_below(z_row, thr=-1.0):
    below = (z_row < thr)
    # compute run lengths
    max_run = 0
    run = 0
    for b in below:
        if b:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

def num_runs_below(z_row, thr=-1.0):
    below = (z_row < thr)
    # count transitions False->True
    return int(np.sum((~below[:-1]) & (below[1:])) + (1 if below[0] else 0))

def extract_features(X):
    Z = per_series_zscore(X)
    feats = []
    for z in Z:
        diff = np.diff(z)
        p01 = np.quantile(z, 0.01)
        p05 = np.quantile(z, 0.05)
        feats.append({
            "min_z": float(np.min(z)),
            "p01_z": float(p01),
            "p05_z": float(p05),
            "frac_below_-1": float(np.mean(z < -1)),
            "frac_below_-2": float(np.mean(z < -2)),
            "frac_below_-3": float(np.mean(z < -3)),
            "longest_run_-1": float(longest_run_below(z, -1)),
            "num_runs_-1": float(num_runs_below(z, -1)),
            "mad_z": float(np.median(np.abs(z - np.median(z)))),
            "iqr_z": float(np.quantile(z, 0.75) - np.quantile(z, 0.25)),
            "mean_abs_diff": float(np.mean(np.abs(diff))),
            "std_diff": float(np.std(diff)),
        })
    return pd.DataFrame(feats)

feat_tr = extract_features(X_train)

In [41]:
def boxplots_by_class(feat_df, y, cols, title):
    fig, axes = plt.subplots(2, 3, figsize=(12,6))
    axes = axes.ravel()
    for ax, col in zip(axes, cols):
        ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
        ax.set_title(col)
        ax.set_xlabel("class (0=non, 1=exo)")
    fig.suptitle(title)
    plt.savefig(f"./plots/engineered_features_box.png")
    plt.close()

key_cols = ["p01_z","frac_below_-2","longest_run_-1","mean_abs_diff","mad_z","min_z"]
boxplots_by_class(feat_tr, y_train, key_cols, "Engineered feature distributions (train)")

def strip_over_box(feat_df, y, col, title):
    x0 = np.zeros(np.sum(y==0))
    x1 = np.ones(np.sum(y==1))
    fig, ax = plt.subplots(figsize=(6,4))
    ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], positions=[0,1], widths=0.5)
    ax.scatter(x0 + 0.05*rng.standard_normal(len(x0)), feat_df.loc[y==0, col], s=10, alpha=0.4)
    ax.scatter(x1 + 0.05*rng.standard_normal(len(x1)), feat_df.loc[y==1, col], s=25, alpha=0.9)
    ax.set_xticks([0,1]); ax.set_xticklabels(["0","1"])
    ax.set_title(title); ax.set_xlabel("class"); ax.set_ylabel(col)
    plt.savefig(f"./plots/p01_z_strip_over_box.png")
    plt.close()

strip_over_box(feat_tr, y_train, "p01_z", "p01_z by class (train)")

C:\Users\haesu\AppData\Local\Temp\ipykernel_17920\1657152811.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
C:\Users\haesu\AppData\Local\Temp\ipykernel_17920\1657152811.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
C:\Users\haesu\AppData\Local\Temp\ipykernel_17920\1657152811.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
C:\Users\haesu\AppData\Local\Temp\ipykernel_17

In [42]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

corr = feat_tr.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation among engineered features (train)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.savefig("./plots/feature_correlation.png")
plt.close()

Xf = feat_tr.to_numpy()
Xf_std = StandardScaler().fit_transform(Xf)
pca = PCA(n_components=2, random_state=0)
X2 = pca.fit_transform(Xf_std)

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(X2[y_train==0,0], X2[y_train==0,1], s=10, alpha=0.4, label="0")
ax.scatter(X2[y_train==1,0], X2[y_train==1,1], s=30, alpha=0.9, label="1")
ax.set_title("PCA of engineered features (train)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
plt.savefig("./plots/feature_pca.png")
plt.close()

In [43]:
# ---------- scaling helpers ----------
def per_series_zscore(X: np.ndarray, eps: float = 1e-8):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    return (X - mu) / (sd + eps)

def per_series_robust(X: np.ndarray, eps: float = 1e-8):
    med = np.median(X, axis=1, keepdims=True)
    q75 = np.quantile(X, 0.75, axis=1, keepdims=True)
    q25 = np.quantile(X, 0.25, axis=1, keepdims=True)
    iqr = q75 - q25
    return (X - med) / (iqr + eps)

# ---------- run-length helpers ----------
def longest_run_below(z_row, thr=-1.0):
    below = (z_row < thr)
    max_run = 0
    run = 0
    for b in below:
        if b:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

def num_runs_below(z_row, thr=-1.0):
    below = (z_row < thr)
    return int(np.sum((~below[:-1]) & (below[1:])) + (1 if below[0] else 0))

# ---------- feature extractor ----------
def extract_features_scaled(X: np.ndarray, scale: str = "zscore") -> pd.DataFrame:
    if scale == "zscore":
        Z = per_series_zscore(X)
    elif scale == "robust":
        Z = per_series_robust(X)
    else:
        raise ValueError("scale must be 'zscore' or 'robust'")

    feats = []
    for z in Z:
        diff = np.diff(z)
        feats.append({
            "min_z": float(np.min(z)),
            "p01_z": float(np.quantile(z, 0.01)),
            "p05_z": float(np.quantile(z, 0.05)),
            "frac_below_-1": float(np.mean(z < -1)),
            "frac_below_-2": float(np.mean(z < -2)),
            "frac_below_-3": float(np.mean(z < -3)),
            "longest_run_-1": float(longest_run_below(z, -1)),
            "num_runs_-1": float(num_runs_below(z, -1)),
            "mad_z": float(np.median(np.abs(z - np.median(z)))),
            "iqr_z": float(np.quantile(z, 0.75) - np.quantile(z, 0.25)),
            "mean_abs_diff": float(np.mean(np.abs(diff))),
            "std_diff": float(np.std(diff)),
            "acf1": float(np.corrcoef(z[:-1], z[1:])[0, 1]) if np.std(z[:-1]) > 0 and np.std(z[1:]) > 0 else 0.0,
            "acf2": float(np.corrcoef(z[:-2], z[2:])[0, 1]) if np.std(z[:-2]) > 0 and np.std(z[2:]) > 0 else 0.0,
        })
    return pd.DataFrame(feats)

feat_tr_z = extract_features_scaled(X_train, scale="zscore")
feat_te_z = extract_features_scaled(X_test,  scale="zscore")

feat_tr_r = extract_features_scaled(X_train, scale="robust")
feat_te_r = extract_features_scaled(X_test,  scale="robust")

In [44]:
def summarize_feature_differences(feat_df: pd.DataFrame, y: np.ndarray, cols):
    rows = []
    for col in cols:
        neg = feat_df.loc[y == 0, col].to_numpy()
        pos = feat_df.loc[y == 1, col].to_numpy()

        med_neg = np.median(neg)
        med_pos = np.median(pos)
        mean_neg = np.mean(neg)
        mean_pos = np.mean(pos)

        # simple pooled-std effect size proxy
        pooled_std = np.sqrt((np.var(neg, ddof=1) + np.var(pos, ddof=1)) / 2)
        effect = (mean_pos - mean_neg) / (pooled_std + 1e-8)

        rows.append({
            "feature": col,
            "median_neg": med_neg,
            "median_pos": med_pos,
            "median_diff": med_pos - med_neg,
            "mean_neg": mean_neg,
            "mean_pos": mean_pos,
            "effect_size_proxy": effect
        })

    out = pd.DataFrame(rows)
    out["abs_effect"] = out["effect_size_proxy"].abs()
    out = out.sort_values("abs_effect", ascending=False).drop(columns="abs_effect")
    return out

key_cols = ["p01_z", "frac_below_-2", "longest_run_-1", "mean_abs_diff", "mad_z", "min_z"]
feature_summary = summarize_feature_differences(feat_tr_z, y_train, key_cols)
print(feature_summary)

def plot_feature_effect_sizes(summary_df: pd.DataFrame, title="Feature differences by class"):
    df = summary_df.copy().sort_values("effect_size_proxy")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(df["feature"], df["effect_size_proxy"])
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_xlabel("Effect size proxy (mean_pos - mean_neg) / pooled std")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f"./plots/feature_effect_sizes.png")
    plt.close()

plot_feature_effect_sizes(feature_summary, "Feature effect size proxies (train)")

          feature  median_neg  median_pos  median_diff   mean_neg   mean_pos  \
3   mean_abs_diff    0.423039    0.243727    -0.179313   0.430096   0.263730   
0           p01_z   -2.082875   -2.469438    -0.386562  -2.097199  -2.852652   
1   frac_below_-2    0.012199    0.022208     0.010009   0.013778   0.021312   
2  longest_run_-1   16.000000   24.000000     8.000000  33.235446  93.216216   
4           mad_z    0.508674    0.582607     0.073933   0.494015   0.540794   
5           min_z   -3.794927   -4.310022    -0.515095  -5.486261  -5.264079   

   effect_size_proxy  
3          -0.942935  
0          -0.738829  
1           0.618549  
2           0.553805  
4           0.262549  
5           0.052974  


In [45]:
def plot_skewed_feature_log_box(feat_df: pd.DataFrame, y: np.ndarray, col="longest_run_-1"):
    vals0 = np.log1p(feat_df.loc[y == 0, col].to_numpy())
    vals1 = np.log1p(feat_df.loc[y == 1, col].to_numpy())

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot([vals0, vals1], labels=["Non-exoplanet (0)", "Exoplanet (1)"])
    ax.set_ylabel(f"log1p({col})")
    ax.set_title(f"Skew-adjusted distribution of {col}")
    plt.tight_layout()
    plt.savefig(f"./plots/longest_run_-1_box.png")
    plt.close()

plot_skewed_feature_log_box(feat_tr_z, y_train, "longest_run_-1")

C:\Users\haesu\AppData\Local\Temp\ipykernel_17920\2468572731.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([vals0, vals1], labels=["Non-exoplanet (0)", "Exoplanet (1)"])


In [46]:
compare_cols = ["p01_z", "frac_below_-2", "mean_abs_diff"]

def compare_scaling_effects(feat_tr_z, feat_tr_r, y, cols):
    zsum = summarize_feature_differences(feat_tr_z, y, cols).set_index("feature")
    rsum = summarize_feature_differences(feat_tr_r, y, cols).set_index("feature")

    comp = pd.DataFrame({
        "zscore": zsum.loc[cols, "effect_size_proxy"],
        "robust": rsum.loc[cols, "effect_size_proxy"]
    })

    x = np.arange(len(cols))
    w = 0.35

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(x - w/2, comp["zscore"], w, label="z-score")
    ax.bar(x + w/2, comp["robust"], w, label="robust")
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(cols, rotation=30, ha="right")
    ax.set_ylabel("Effect size proxy")
    ax.set_title("Feature effect sizes under z-score vs robust scaling")
    ax.legend()
    plt.tight_layout()
    plt.savefig("./plots/scaling_effect_size_comparison.png")
    plt.close()

compare_scaling_effects(feat_tr_z, feat_tr_r, y_train, compare_cols)

In [47]:
def plot_selected_positive_lightcurves(
    X: np.ndarray,
    y: np.ndarray,
    sort_by="p01_z",
    feat_df=None,
    n_keep=12,
    ncols=4,
    random_state=42,
):
    Z = per_series_zscore(X)
    pos_idx = np.where(y == 1)[0]

    if feat_df is None or sort_by is None:
        raise ValueError("feat_df and sort_by must be provided for curated selection.")

    # Sort positives by chosen feature
    order = np.argsort(feat_df.loc[pos_idx, sort_by].to_numpy())
    pos_idx_sorted = pos_idx[order]

    n_pos = len(pos_idx_sorted)
    n_keep = min(n_keep, n_pos)

    # Split into thirds: low / middle / high
    thirds = np.array_split(pos_idx_sorted, 3)

    # Allocate roughly evenly across thirds
    base = n_keep // 3
    remainder = n_keep % 3
    counts = [base, base, base]
    for i in range(remainder):
        counts[i] += 1

    rng = np.random.default_rng(random_state)
    selected = []

    for group, k in zip(thirds, counts):
        if len(group) <= k:
            selected.extend(group.tolist())
        else:
            # evenly spaced picks from each third, instead of purely random
            pick_idx = np.linspace(0, len(group) - 1, k, dtype=int)
            selected.extend(group[pick_idx].tolist())

    selected = np.array(selected)

    # Plot
    nrows = int(np.ceil(len(selected) / ncols))
    t = np.arange(X.shape[1])

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(16, 12),
        sharex=True,
        sharey=True
    )
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, selected):
        ax.plot(t, Z[idx], linewidth=0.8)
        ax.set_title(f"idx={idx}\n{sort_by}={feat_df.loc[idx, sort_by]:.2f}", fontsize=8)
        ax.set_xlabel("time")
        ax.set_ylabel("z")

    # Hide unused axes
    for ax in axes[len(selected):]:
        ax.axis("off")

    fig.suptitle("Representative positive-class light curves (z-scored)", y=0.995)
    plt.tight_layout()
    plt.savefig("./plots/selected_positive_lightcurves.png", dpi=200, bbox_inches="tight")
    plt.close()

plot_selected_positive_lightcurves(X_train, y_train, sort_by="p01_z", feat_df=feat_tr_z, n_keep=12, ncols=4)

In [48]:
def overlay_train_test_hist(feat_train: pd.DataFrame, feat_test: pd.DataFrame, col: str, bins=40):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(feat_train[col], bins=bins, alpha=0.6, label="Train")
    ax.hist(feat_test[col],  bins=bins, alpha=0.6, label="Test")
    ax.set_title(f"Train vs test distribution: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend()
    plt.savefig(f"./plots/train_test_{col}_hist.png")
    plt.close()

for col in ["p01_z", "frac_below_-2", "mean_abs_diff"]:
    overlay_train_test_hist(feat_tr_z, feat_te_z, col)

In [49]:
from PIL import Image
import matplotlib.pyplot as plt
import os

image_paths = [
    "./plots/selected_positive_lightcurves.png",
    "./plots/engineered_features_box.png",
    "./plots/feature_correlation.png",
    "./plots/feature_effect_sizes.png",
    "./plots/feature_pca.png",
    "./plots/lightcurves_panel.png",
    "./plots/longest_run_-1_box.png",
    "./plots/p01_z_strip_over_box.png",
    "./plots/scaling_effect_size_comparison.png",
    "./plots/train_test_frac_below_-2_hist.png",
    "./plots/train_test_mean_abs_diff_hist.png",
    "./plots/train_test_p01_z_hist.png",
]

titles = [
    "All Positive Light Curves",
    "Engineered Features Boxplots",
    "Feature Correlation",
    "Feature Effect Sizes",
    "Feature PCA",
    "Light Curves Panel",
    "log1p(Longest Run) Boxplot",
    "p01_z Strip+Box",
    "Scaling Effect Size Comparison",
    "Train/Test frac_below_-2",
    "Train/Test mean_abs_diff",
    "Train/Test p01_z",
]

missing = [p for p in image_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"Missing image files: {missing}")

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.ravel()

for ax, path, title in zip(axes, image_paths, titles):
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("./plots/eda_contact_sheet.png", dpi=200, bbox_inches="tight")
plt.close()

# Part 2: Feature selection and preprocessing

We finalized per-star z-score normalization as the main preprocessing step. This was chosen because raw flux values varied dramatically across stars, making absolute magnitude unreliable for comparison. Z-scoring preserved relative light-curve shape and dimming behavior, which is what our engineered features were meant to capture.

We also compared z-score against robust scaling (median/IQR) as a sanity check. The key feature trends kept the same directional behavior under both methods, so we kept z-score for simplicity and consistency.

From the broader engineered feature pool, we first examined pairwise correlations to identify redundancy. This showed several near-duplicate groups, such as `mad_z` with `iqr_z`, `acf1` with `acf2`, and strong overlap among the threshold-based dip-frequency features. Based on this, we pruned to a smaller candidate set that kept one representative from each correlated group.

We then ranked features by a simple effect size proxy comparing exoplanet vs non-exoplanet classes. This showed that both local structure/smoothness features (`acf1`, `mean_abs_diff`, `std_diff`) and transit-motivated dip features (`p01_z`, `frac_below_-2`, `longest_run_-1`) carried signal.

To make the selection less subjective, we ran L1-regularized logistic regression with 5-fold stratified cross-validation as a stability check. Features consistently selected across folds were treated as the strongest candidates.

The features selected in all 5 folds were: `acf1`, `mad_z`, `longest_run_-1`, `frac_below_-2`, `p01_z`, and `mean_abs_diff`. By contrast, `std_diff` and `num_runs_-1` were selected inconsistently, so they were excluded from the final core set.

This left us with a compact final feature set that captures:
- short-lag structure / smoothness: `acf1`, `mean_abs_diff`
- overall robust variability: `mad_z`
- depth, frequency, and duration of low-flux behavior: `p01_z`, `frac_below_-2`, `longest_run_-1`

Overall, the feature selection process supported the idea that exoplanet-labeled light curves differ not only by having deeper or longer low-flux excursions, but also by exhibiting smoother, more structured local behavior than many negative examples.

In [50]:
candidate_cols = [
    "min_z",
    "p01_z",
    "p05_z",
    "frac_below_-1",
    "frac_below_-2",
    "frac_below_-3",
    "longest_run_-1",
    "num_runs_-1",
    "mad_z",
    "iqr_z",
    "mean_abs_diff",
    "std_diff",
    "acf1",
    "acf2",
]

candidate_df = feat_tr_z[candidate_cols].copy()
candidate_df.head()

,min_z,p01_z,p05_z,frac_below_-1,frac_below_-2,frac_below_-3,longest_run_-1,num_runs_-1,mad_z,iqr_z,mean_abs_diff,std_diff,acf1,acf2
0,-6.620421,-2.143195,-1.532662,0.154833,0.015014,0.001877,24.0,147.0,0.690029,1.381925,0.351541,0.551851,0.847769,0.721402
1,-6.063865,-4.754909,-1.277299,0.060369,0.037222,0.029403,11.0,56.0,0.390156,0.776812,0.332721,0.634420,0.798803,0.590351
2,-3.133562,-2.542234,-1.816857,0.115421,0.035346,0.001877,52.0,36.0,0.504307,1.021813,0.147196,0.298306,0.955496,0.912262
3,-2.555287,-2.162253,-1.670249,0.177041,0.026275,0.000000,198.0,54.0,0.706338,1.430985,0.128399,0.203164,0.979337,0.967220
4,-4.408515,-2.462599,-1.842500,0.162652,0.028151,0.002502,169.0,44.0,0.728054,1.433434,0.126879,0.303894,0.953776,0.909577


In [51]:
def high_correlation_pairs(df: pd.DataFrame, threshold: float = 0.80) -> pd.DataFrame:
    corr = df.corr(numeric_only=True)
    rows = []
    cols = corr.columns.tolist()

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            val = corr.iloc[i, j]
            if abs(val) >= threshold:
                rows.append({
                    "feature_1": cols[i],
                    "feature_2": cols[j],
                    "correlation": val
                })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.reindex(out["correlation"].abs().sort_values(ascending=False).index)
    return out

high_corr = high_correlation_pairs(candidate_df, threshold=0.80)
print(high_corr.to_string(index=False))

    feature_1     feature_2  correlation
        mad_z         iqr_z     0.986815
     std_diff          acf1    -0.970432
     std_diff          acf2    -0.954342
frac_below_-1         mad_z     0.938715
         acf1          acf2     0.938013
frac_below_-1         iqr_z     0.934442
        p05_z frac_below_-1    -0.867976
  num_runs_-1 mean_abs_diff     0.860835
        p05_z         mad_z    -0.812535
        p05_z         iqr_z    -0.804235


In [52]:
def summarize_feature_differences(feat_df: pd.DataFrame, y: np.ndarray, cols):
    rows = []
    for col in cols:
        neg = feat_df.loc[y == 0, col].to_numpy()
        pos = feat_df.loc[y == 1, col].to_numpy()

        mean_neg = np.mean(neg)
        mean_pos = np.mean(pos)
        med_neg = np.median(neg)
        med_pos = np.median(pos)

        pooled_std = np.sqrt((np.var(neg, ddof=1) + np.var(pos, ddof=1)) / 2)
        effect = (mean_pos - mean_neg) / (pooled_std + 1e-8)

        rows.append({
            "feature": col,
            "mean_neg": mean_neg,
            "mean_pos": mean_pos,
            "median_neg": med_neg,
            "median_pos": med_pos,
            "effect_size_proxy": effect,
            "abs_effect": abs(effect),
        })

    out = pd.DataFrame(rows).sort_values("abs_effect", ascending=False)
    return out.drop(columns="abs_effect")

feature_signal = summarize_feature_differences(candidate_df, y_train, candidate_cols)
print(feature_signal.to_string(index=False))

       feature   mean_neg  mean_pos  median_neg  median_pos  effect_size_proxy
          acf1   0.610794  0.837840    0.658233    0.847769           1.088409
      std_diff   0.819916  0.507450    0.825770    0.551851          -1.059589
          acf2   0.503501  0.733196    0.501434    0.745301           0.976957
 mean_abs_diff   0.430096  0.263730    0.423039    0.243727          -0.942935
         p01_z  -2.097199 -2.852652   -2.082875   -2.469438          -0.738829
   num_runs_-1 123.426931 76.513514  115.000000   72.000000          -0.692149
 frac_below_-2   0.013778  0.021312    0.012199    0.022208           0.618549
longest_run_-1  33.235446 93.216216   16.000000   24.000000           0.553805
 frac_below_-3   0.003471  0.006501    0.001251    0.003128           0.447499
 frac_below_-1   0.099378  0.122133    0.103222    0.137942           0.413875
         p05_z  -1.280100 -1.397753   -1.345194   -1.532662          -0.281936
         mad_z   0.494015  0.540794    0.508674    0

In [53]:
# First-pass pruned set based on correlation groups + interpretability
pruned_cols = [
    "p01_z",          # representative of low-tail depth
    "frac_below_-2",  # representative of deep-dip frequency
    "longest_run_-1", # representative of dip duration
    "mean_abs_diff",  # representative of roughness
    "mad_z",          # representative of robust dispersion
    "acf1",           # optional structure feature
    "std_diff",       # optional secondary roughness feature
    "num_runs_-1",    # optional secondary dip-count feature
]

pruned_df = feat_tr_z[pruned_cols].copy()
print("Pruned feature set:")
print(pruned_df.columns.tolist())

Pruned feature set:
['p01_z', 'frac_below_-2', 'longest_run_-1', 'mean_abs_diff', 'mad_z', 'acf1', 'std_diff', 'num_runs_-1']


In [54]:
scaler = StandardScaler()
X_pruned_scaled = scaler.fit_transform(pruned_df)

print("Scaled feature matrix shape:", X_pruned_scaled.shape)

Scaled feature matrix shape: (5087, 8)


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

def l1_feature_selection_stability(X, y, feature_names, C=0.1, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    coef_rows = []
    selected_counts = pd.Series(0, index=feature_names, dtype=int)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, y_tr = X[train_idx], y[train_idx]

        clf = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=random_state
        )
        clf.fit(X_tr, y_tr)

        coefs = pd.Series(clf.coef_[0], index=feature_names, name=f"fold_{fold}")
        coef_rows.append(coefs)

        selected_counts += (coefs != 0).astype(int)

    coef_df = pd.concat(coef_rows, axis=1)
    summary = pd.DataFrame({
        "feature": feature_names,
        "selected_in_folds": selected_counts.values,
        "selection_rate": selected_counts.values / n_splits,
        "mean_coef": coef_df.mean(axis=1).values,
        "std_coef": coef_df.std(axis=1).values,
    }).sort_values(["selection_rate", "selected_in_folds", "mean_coef"], ascending=[False, False, False])

    return coef_df, summary

coef_df, coef_summary = l1_feature_selection_stability(
    X_pruned_scaled,
    y_train,
    feature_names=pruned_cols,
    C=0.1,
    n_splits=5
)

print(coef_summary.to_string(index=False))

       feature  selected_in_folds  selection_rate  mean_coef  std_coef
          acf1                  5             1.0   0.972302  1.042215
         mad_z                  5             1.0   0.231584  0.177184
longest_run_-1                  5             1.0   0.102139  0.082379
 frac_below_-2                  5             1.0   0.045360  0.126105
         p01_z                  5             1.0  -0.722141  0.107756
 mean_abs_diff                  5             1.0  -0.799903  0.455805
      std_diff                  3             0.6   0.469355  0.445929
   num_runs_-1                  2             0.4   0.130698  0.197219


c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', 

In [56]:
final_cols = coef_summary.loc[coef_summary["selected_in_folds"] >= 3, "feature"].tolist()

# If too many survive, keep top 5-6 by selection_rate then abs(mean_coef)
if len(final_cols) > 6:
    tmp = coef_summary.copy()
    tmp["abs_mean_coef"] = tmp["mean_coef"].abs()
    final_cols = (
        tmp.sort_values(["selection_rate", "abs_mean_coef"], ascending=[False, False])
           .head(6)["feature"]
           .tolist()
    )

print("Final selected features:")
print(final_cols)

Final selected features:
['acf1', 'mean_abs_diff', 'p01_z', 'mad_z', 'longest_run_-1', 'frac_below_-2']


# Part 3: Final modeling protocol setup

We finalized the modeling dataset using the six selected engineered features: `acf1`, `mean_abs_diff`, `p01_z`, `mad_z`, `longest_run_-1`, and `frac_below_-2`. This fixed the feature space before inferential or predictive modeling so that later comparisons would all use the same inputs.

The resulting feature matrices had shapes (5087, 6) for training and (570, 6) for test, confirming that the downstream models would operate on a compact, low-dimensional representation rather than the full raw light-curve sequence.

We kept both unscaled and standardized versions of the final feature matrix. Standardization was applied using statistics fit on the training set only, and the scaled results had near-zero means and unit variance across all six features, as expected. This standardized representation is especially important for logistic regression so that coefficient magnitudes are comparable and regularization behaves properly.

To make validation stable and fair under extreme class imbalance, we fixed a 5-fold stratified cross-validation protocol on the training set. The folds were balanced as well as possible given the small number of positives, with validation folds containing 7-8 positive examples each. This is a critical constraint of the dataset: each fold still contains very few positives, so performance estimates are expected to have noticeable variance.

The provided test set was left fully held out during this stage. This means all feature selection, coefficient analysis, model comparison, calibration, and threshold tuning can be done using the training set and its fixed CV structure, with the test set reserved only for final evaluation.

We also recorded baseline reference values before fitting any models. The training positive rate was 0.727%, and the test positive rate was 0.877%. An always-negative classifier would therefore achieve 99.27% accuracy on train and 99.12% accuracy on test, which reinforces that raw accuracy is not meaningful for this problem. Similarly, the positive base rate (~0.7%) serves as a “random precision” baseline: a useful model should substantially exceed this precision level.

Finally, we defined the reusable evaluation framework in advance. We set up metric helpers for:
- probability-based evaluation (`PR-AUC`, `ROC-AUC`)
- threshold-based evaluation (precision, recall, F1, confusion matrix)
- and threshold search under a minimum precision constraint (e.g. maximizing recall while maintaining precision ≥ 0.10).

Overall, this step froze the modeling protocol before any inferential or predictive results were generated. That reduces the risk of changing folds, metrics, or preprocessing mid-analysis and makes the later comparisons across models directly comparable.

## Main takeaway

* At this point, the experimental setup is fixed: the feature set, preprocessing pipeline, validation folds, and evaluation logic are all locked in. This means the next stages -- interpreting coefficients in the inferential model and comparing predictive models -- can focus entirely on model behavior rather than setup decisions.


In [57]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, average_precision_score, roc_auc_score

# Final selected feature set from Part 2
final_cols = [
    "acf1",
    "mean_abs_diff",
    "p01_z",
    "mad_z",
    "longest_run_-1",
    "frac_below_-2",
]

# Build final train/test feature matrices
X_train_final = feat_tr_z[final_cols].copy()
X_test_final  = feat_te_z[final_cols].copy()

print("Final feature columns:")
print(final_cols)
print()

print("Train feature matrix shape:", X_train_final.shape)
print("Test feature matrix shape :", X_test_final.shape)
print()

# --------------------------------------------------
# Unscaled and scaled versions
# --------------------------------------------------

# Keep raw tabular feature values (useful for tree-based models)
X_train_unscaled = X_train_final.to_numpy(dtype=float)
X_test_unscaled  = X_test_final.to_numpy(dtype=float)

# Standardized version (fit on train only)
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_unscaled)
X_test_scaled  = scaler_final.transform(X_test_unscaled)

print("Scaled train matrix shape:", X_train_scaled.shape)
print("Scaled test matrix shape :", X_test_scaled.shape)
print()

# Optional: inspect feature means/stds after scaling
scaled_df = pd.DataFrame(X_train_scaled, columns=final_cols)
print("Approx. scaled train feature means:")
print(scaled_df.mean().round(4).to_string())
print()
print("Approx. scaled train feature stds:")
print(scaled_df.std(ddof=0).round(4).to_string())
print()

# --------------------------------------------------
# Fixed cross-validation protocol
# --------------------------------------------------

n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_splits = list(cv.split(X_train_scaled, y_train))

print(f"Number of CV folds: {len(cv_splits)}")
for i, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
    y_tr_fold = y_train[tr_idx]
    y_val_fold = y_train[val_idx]
    print(
        f"Fold {i}: "
        f"train size={len(tr_idx)}, val size={len(val_idx)}, "
        f"train positives={y_tr_fold.sum()}, val positives={y_val_fold.sum()}"
    )

train_pos_rate = float(np.mean(y_train))
test_pos_rate = float(np.mean(y_test))

always_negative_train_acc = float(np.mean(y_train == 0))
always_negative_test_acc = float(np.mean(y_test == 0))

print("Baseline reference values")
print(f"Train positive rate: {train_pos_rate:.6f} ({train_pos_rate*100:.3f}%)")
print(f"Test positive rate : {test_pos_rate:.6f} ({test_pos_rate*100:.3f}%)")
print(f"Always-negative train accuracy: {always_negative_train_acc:.6f}")
print(f"Always-negative test accuracy : {always_negative_test_acc:.6f}")
print(f"Random precision baseline (train) ~= positive rate = {train_pos_rate:.6f}")
print()

# --------------------------------------------------
# Metric helper functions for later sections
# --------------------------------------------------

def compute_threshold_metrics(y_true, y_prob, threshold=0.5):
    """
    Compute threshold-based metrics from predicted probabilities.
    """
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }
    return out

def compute_prob_metrics(y_true, y_prob):
    """
    Compute probability-based metrics that do not require a threshold.
    """
    out = {
        "pr_auc": average_precision_score(y_true, y_prob),
    }

    # ROC-AUC is only defined if both classes are present
    if len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        out["roc_auc"] = np.nan

    return out

def find_threshold_for_min_precision(y_true, y_prob, min_precision=0.10, n_steps=1000):
    """
    Find the threshold that maximizes recall while keeping precision >= min_precision.
    Simple grid search over [0, 1].
    """
    thresholds = np.linspace(0.0, 1.0, n_steps)
    best = None

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        if prec >= min_precision:
            candidate = {
                "threshold": t,
                "precision": prec,
                "recall": rec,
                "f1": f1,
            }

            if best is None or candidate["recall"] > best["recall"]:
                best = candidate

    return best

print("Metric helpers ready:")
print("- compute_prob_metrics(y_true, y_prob)")
print("- compute_threshold_metrics(y_true, y_prob, threshold=...)")
print("- find_threshold_for_min_precision(y_true, y_prob, min_precision=0.10)")

Final feature columns:
['acf1', 'mean_abs_diff', 'p01_z', 'mad_z', 'longest_run_-1', 'frac_below_-2']

Train feature matrix shape: (5087, 6)
Test feature matrix shape : (570, 6)

Scaled train matrix shape: (5087, 6)
Scaled test matrix shape : (570, 6)

Approx. scaled train feature means:
acf1              0.0
mean_abs_diff     0.0
p01_z             0.0
mad_z            -0.0
longest_run_-1    0.0
frac_below_-2     0.0

Approx. scaled train feature stds:
acf1              1.0
mean_abs_diff     1.0
p01_z             1.0
mad_z             1.0
longest_run_-1    1.0
frac_below_-2     1.0

Number of CV folds: 5
Fold 1: train size=4069, val size=1018, train positives=29, val positives=8
Fold 2: train size=4069, val size=1018, train positives=29, val positives=8
Fold 3: train size=4070, val size=1017, train positives=30, val positives=7
Fold 4: train size=4070, val size=1017, train positives=30, val positives=7
Fold 5: train size=4070, val size=1017, train positives=30, val positives=7
Baseline

# Part 4: Inferntial Analysis

For the inferential analysis, we used a balanced L2-regularized logistic regression on the six finalized standardized features: `acf1`, `mean_abs_diff`, `p01_z`, `mad_z`, `longest_run_-1`, and `frac_below_-2`. This model was chosen because it allows direct coefficient interpretation while remaining more stable than the earlier L1 model that was used primarily for feature selection.

We first fit the model across the fixed 5-fold stratified cross-validation splits from Part 3 and collected coefficients fold by fold. This allowed us to evaluate coefficient stability before interpreting the full-train fit. The most stable feature by far was `p01_z`, which remained negative in all five folds with relatively low variability, indicating that deeper low-tail flux behavior is consistently associated with exoplanet-labeled stars.

`acf1` was also directionally stable, remaining positive in every fold, although its coefficient magnitude varied substantially from fold to fold. This suggests that short-lag smoothness or continuity is likely a real part of the positive-class signal, but that its estimated strength is sensitive to the small number of positive examples in each fold.

`mean_abs_diff` was negative in most folds and had a relatively large point estimate overall, suggesting that exoplanet-labeled light curves tend to be less jagged locally. However, one fold showed a slight sign flip, so this feature was treated as meaningful but less stable than `p01_z`.

By contrast, `mad_z`, `longest_run_-1`, and `frac_below_-2` appeared much weaker in the multivariable inferential model than they did in the earlier EDA. Although they showed useful class differences when examined on their own, their coefficients were small and sign-inconsistent once the stronger selected features were included together. This suggests that much of their signal overlaps with the stronger predictors.

After the fold-wise stability check, we fit the same logistic regression model on the full training set to obtain the final coefficient estimates. In that model, the strongest coefficients were:

- `p01_z`: coefficient ≈ -0.758, odds ratio ≈ 0.47
- `mean_abs_diff`: coefficient ≈ -0.696, odds ratio ≈ 0.50
- `acf1`: coefficient ≈ +0.461, odds ratio ≈ 1.59

These results indicate that deeper lower-tail dimming, lower local roughness, and greater short-lag smoothness are the strongest conditional associations in the final inferential model.

We then merged the CV stability summary and the full-train fit into a single inferential summary table. This let us compare sign consistency, coefficient variability, and final full-train effect size side by side, rather than relying on only one fitted model.

To add uncertainty context without leaning on classical p-values, we also ran a bootstrap analysis of the full logistic model. This confirmed that `p01_z` was the most robust inferential feature: its bootstrap interval remained fully below zero, supporting a stable negative association. By contrast, the bootstrap intervals for `acf1` and `mean_abs_diff` were wider and crossed zero, meaning that while their point estimates were substantial, their exact effects should be interpreted more cautiously. The remaining features had wide intervals centered close to zero, reinforcing that they are weaker independent contributors in the multivariable setting.

Overall, the inferential analysis suggests that the clearest independent distinguishing characteristic of exoplanet-labeled light curves is deeper low-tail dimming, with local smoothness and short-lag structure providing additional but less certain supporting information. Features tied more directly to run length and low-flux frequency still mattered descriptively in EDA, but did not remain strong conditional signals once the stronger predictors were modeled jointly.

## Main Part 4 takeaway

The inferential model showed that not all engineered features that look useful in EDA remain equally important once modeled together. The most robust conditional signal is deeper lower-tail flux behavior (`p01_z`), while smoothness-related features (`acf1`, `mean_abs_diff`) appear meaningful but more uncertain. This gives a more nuanced picture than the original dataset description alone, which mainly emphasizes repeated dimming.


In [58]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# Step 4.1: Fold-wise inferential logistic regression
# --------------------------------------------------

# Assumes these already exist from Part 3:
# - X_train_scaled
# - y_train
# - cv_splits
# - final_cols

def collect_cv_logistic_coefficients(
    X,
    y,
    cv_splits,
    feature_names,
    C=1.0,
    random_state=42
):
    """
    Fit balanced L2 logistic regression on each CV training fold
    and collect coefficients for stability analysis.
    """
    coef_rows = []
    intercept_rows = []

    for fold_idx, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
        X_tr = X[tr_idx]
        y_tr = y[tr_idx]

        clf = LogisticRegression(
            penalty="l2",
            C=C,
            solver="liblinear",   # stable for small-ish tabular problems
            class_weight="balanced",
            max_iter=5000,
            random_state=random_state
        )

        clf.fit(X_tr, y_tr)

        fold_coef = pd.Series(clf.coef_[0], index=feature_names, name=f"fold_{fold_idx}")
        coef_rows.append(fold_coef)

        intercept_rows.append({
            "fold": fold_idx,
            "intercept": float(clf.intercept_[0])
        })

    coef_df = pd.concat(coef_rows, axis=1)  # rows=features, cols=folds
    intercept_df = pd.DataFrame(intercept_rows)

    return coef_df, intercept_df


def summarize_cv_coefficients(coef_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize coefficient stability across folds.
    """
    values = coef_df.to_numpy()

    mean_coef = coef_df.mean(axis=1)
    std_coef = coef_df.std(axis=1)

    # Fraction of folds with positive / negative coefficient
    pos_rate = (coef_df > 0).mean(axis=1)
    neg_rate = (coef_df < 0).mean(axis=1)

    # Sign consistency: all positive, all negative, or mixed
    sign_consistency = []
    for _, row in coef_df.iterrows():
        if np.all(row > 0):
            sign_consistency.append("all_positive")
        elif np.all(row < 0):
            sign_consistency.append("all_negative")
        else:
            sign_consistency.append("mixed")

    summary = pd.DataFrame({
        "feature": coef_df.index,
        "mean_coef": mean_coef.values,
        "std_coef": std_coef.values,
        "pos_rate": pos_rate.values,
        "neg_rate": neg_rate.values,
        "sign_consistency": sign_consistency,
        "abs_mean_coef": np.abs(mean_coef.values),
    })

    summary = summary.sort_values("abs_mean_coef", ascending=False).drop(columns="abs_mean_coef")
    return summary


# Run the fold-wise coefficient collection
coef_df, intercept_df = collect_cv_logistic_coefficients(
    X=X_train_scaled,
    y=y_train,
    cv_splits=cv_splits,
    feature_names=final_cols,
    C=1.0
)

coef_summary = summarize_cv_coefficients(coef_df)

print("Fold-wise coefficient matrix:")
print(coef_df.round(4).to_string())
print()

print("Intercepts by fold:")
print(intercept_df.round(4).to_string(index=False))
print()

print("Coefficient stability summary:")
print(coef_summary.round(4).to_string(index=False))

Fold-wise coefficient matrix:
                fold_1  fold_2  fold_3  fold_4  fold_5
acf1            0.2544  0.1491  1.8601  0.3380  0.2896
mean_abs_diff  -0.8356 -0.9505  0.0717 -0.6780 -0.7931
p01_z          -0.8187 -0.6242 -0.7270 -0.7380 -0.8705
mad_z           0.2881  0.5472 -0.0059  0.2623  0.2872
longest_run_-1  0.0907 -0.1547  0.1185  0.0325  0.0953
frac_below_-2   0.0084  0.2298 -0.1823  0.0658  0.0476

Intercepts by fold:
 fold  intercept
    1    -0.9913
    2    -0.8915
    3    -1.4275
    4    -0.8376
    5    -1.0511

Coefficient stability summary:
       feature  mean_coef  std_coef  pos_rate  neg_rate sign_consistency
         p01_z    -0.7557    0.0943       0.0       1.0     all_negative
 mean_abs_diff    -0.6371    0.4081       0.2       0.8            mixed
          acf1     0.5782    0.7199       1.0       0.0     all_positive
         mad_z     0.2758    0.1959       0.8       0.2            mixed
longest_run_-1     0.0365    0.1115       0.8       0.2          

c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated

In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

# --------------------------------------------------
# Part 4.2: Final inferential logistic regression
# --------------------------------------------------

def fit_final_inferential_logistic(X, y, feature_names, C=1.0, random_state=42):
    """
    Fit the final balanced L2 logistic regression model
    on the full training set.
    """
    clf = LogisticRegression(
        penalty="l2",
        C=C,
        solver="liblinear",
        class_weight="balanced",
        max_iter=5000,
        random_state=random_state
    )
    clf.fit(X, y)

    coef = clf.coef_[0]
    intercept = float(clf.intercept_[0])

    coef_table = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coef,
        "odds_ratio": np.exp(coef),
        "abs_coefficient": np.abs(coef),
    }).sort_values("abs_coefficient", ascending=False).drop(columns="abs_coefficient")

    return clf, intercept, coef_table


def plot_final_coefficients(coef_table: pd.DataFrame, title="Final inferential logistic coefficients"):
    """
    Horizontal bar plot of final coefficients.
    """
    df = coef_table.copy().sort_values("coefficient")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(df["feature"], df["coefficient"])
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_xlabel("Coefficient (log-odds change per +1 SD)")
    ax.set_title(title)
    plt.savefig("./plots/final_logistic_coefficients.png")
    plt.close()


# Fit final model on all training data
final_logit_model, final_intercept, final_coef_table = fit_final_inferential_logistic(
    X=X_train_scaled,
    y=y_train,
    feature_names=final_cols,
    C=1.0
)

print("Final inferential logistic regression intercept:")
print(round(final_intercept, 6))
print()

print("Final coefficient table:")
print(final_coef_table.round(6).to_string(index=False))
print()

# Plot coefficients
plot_final_coefficients(final_coef_table, title="Final inferential logistic coefficients (full train)")

c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Final inferential logistic regression intercept:
-0.972196

Final coefficient table:
       feature  coefficient  odds_ratio
         p01_z    -0.758292    0.468466
 mean_abs_diff    -0.695522    0.498814
          acf1     0.460751    1.585264
         mad_z     0.278186    1.320731
longest_run_-1     0.049682    1.050937
 frac_below_-2     0.027999    1.028394



In [60]:
def build_inferential_summary_table(coef_summary: pd.DataFrame, final_coef_table: pd.DataFrame) -> pd.DataFrame:
    """
    Merge CV stability summary with full-train coefficient results.
    """
    left = coef_summary.copy()
    right = final_coef_table.copy()

    merged = left.merge(right, on="feature", how="left", suffixes=("_cv", "_full"))

    # Reorder columns for readability
    merged = merged[
        [
            "feature",
            "mean_coef",
            "std_coef",
            "pos_rate",
            "neg_rate",
            "sign_consistency",
            "coefficient",
            "odds_ratio",
        ]
    ]

    # Sort by absolute full-train coefficient
    merged["abs_full_coef"] = merged["coefficient"].abs()
    merged = merged.sort_values("abs_full_coef", ascending=False).drop(columns="abs_full_coef")

    return merged

inferential_summary = build_inferential_summary_table(coef_summary, final_coef_table)

print("Final inferential summary table:")
print(inferential_summary.round(6).to_string(index=False))

Final inferential summary table:
       feature  mean_coef  std_coef  pos_rate  neg_rate sign_consistency  coefficient  odds_ratio
         p01_z  -0.755665  0.094317       0.0       1.0     all_negative    -0.758292    0.468466
 mean_abs_diff  -0.637115  0.408060       0.2       0.8            mixed    -0.695522    0.498814
          acf1   0.578225  0.719937       1.0       0.0     all_positive     0.460751    1.585264
         mad_z   0.275788  0.195867       0.8       0.2            mixed     0.278186    1.320731
longest_run_-1   0.036465  0.111456       0.8       0.2            mixed     0.049682    1.050937
 frac_below_-2   0.033871  0.147458       0.8       0.2            mixed     0.027999    1.028394


In [61]:
def label_inferential_strength(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    labels = []

    for _, row in out.iterrows():
        abs_coef = abs(row["coefficient"])
        sign_consistent = row["sign_consistency"] in {"all_positive", "all_negative"}

        if sign_consistent and abs_coef >= 0.5:
            labels.append("strong")
        elif abs_coef >= 0.15:
            labels.append("moderate")
        else:
            labels.append("weak")

    out["interpretive_strength"] = labels
    return out

inferential_summary_labeled = label_inferential_strength(inferential_summary)

print(inferential_summary_labeled.round(6).to_string(index=False))

       feature  mean_coef  std_coef  pos_rate  neg_rate sign_consistency  coefficient  odds_ratio interpretive_strength
         p01_z  -0.755665  0.094317       0.0       1.0     all_negative    -0.758292    0.468466                strong
 mean_abs_diff  -0.637115  0.408060       0.2       0.8            mixed    -0.695522    0.498814              moderate
          acf1   0.578225  0.719937       1.0       0.0     all_positive     0.460751    1.585264              moderate
         mad_z   0.275788  0.195867       0.8       0.2            mixed     0.278186    1.320731              moderate
longest_run_-1   0.036465  0.111456       0.8       0.2            mixed     0.049682    1.050937                  weak
 frac_below_-2   0.033871  0.147458       0.8       0.2            mixed     0.027999    1.028394                  weak


In [ ]:

def bootstrap_logistic_coefficients(X, y, feature_names, n_boot=200, C=1.0, random_state=42):
    rng = np.random.default_rng(random_state)
    n = len(y)

    coef_samples = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        X_b = X[idx]
        y_b = y[idx]

        # skip degenerate bootstrap samples with only one class
        if len(np.unique(y_b)) < 2:
            continue

        clf = LogisticRegression(
            C=C,
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            random_state=random_state
        )
        clf.fit(X_b, y_b)
        coef_samples.append(clf.coef_[0])

    coef_samples = np.array(coef_samples)

    rows = []
    for j, feat in enumerate(feature_names):
        vals = coef_samples[:, j]
        rows.append({
            "feature": feat,
            "boot_mean": np.mean(vals),
            "boot_std": np.std(vals, ddof=1),
            "ci_2.5": np.quantile(vals, 0.025),
            "ci_97.5": np.quantile(vals, 0.975),
        })

    return pd.DataFrame(rows)

boot_coef_summary = bootstrap_logistic_coefficients(
    X_train_scaled,
    y_train,
    feature_names=final_cols,
    n_boot=200
)

print(boot_coef_summary.round(6).to_string(index=False))

       feature  boot_mean  boot_std    ci_2.5   ci_97.5
          acf1   0.774781  0.925244 -0.593158  2.555563
 mean_abs_diff  -0.526932  0.673447 -1.661768  0.778712
         p01_z  -0.804789  0.183457 -1.192612 -0.460757
         mad_z   0.294990  0.324522 -0.245144  0.993794
longest_run_-1   0.044536  0.170666 -0.365731  0.348978
 frac_below_-2  -0.011701  0.196743 -0.373615  0.341363


# Part 5: Predictive Modeling

For the predictive modeling section, we compared three models on the same finalized six-feature representation: logistic regression, random forest, and gradient boosting. This model set was chosen to provide meaningful variation while staying aligned with the engineered tabular structure of the data: logistic regression served as the baseline and inferentially interpretable model, while random forest and gradient boosting tested whether nonlinear ensemble methods could improve rare-event detection.

We evaluated all three models under the same fixed 5-fold stratified cross-validation protocol established earlier, using the same training folds and the same feature set. The primary metric was PR-AUC, since the positive class was under 1% and accuracy would be misleading. We also tracked ROC-AUC and threshold-based precision, recall, and F1 at the default threshold of 0.5.

The strongest predictive result was that logistic regression clearly outperformed both tree-based ensemble models on this feature set. Logistic regression achieved the highest mean PR-AUC (about 0.106) and the highest mean ROC-AUC (about 0.796), while also being the only model that recovered any positives at the default threshold of 0.5.

In contrast, both Random Forest and Gradient Boosting underperformed. Random Forest had substantially lower PR-AUC and ROC-AUC than logistic regression, and it predicted no positives at all at threshold 0.5 in any fold. Gradient Boosting performed somewhat better than Random Forest in ROC-AUC but still had much lower PR-AUC than logistic regression and also predicted no positives at threshold 0.5.

Going in, we expected that the nonlinear ensemble models might improve predictive performance, especially if there were interactions or nonlinear boundaries in the engineered feature space. Instead, the opposite happened: the simpler linear logistic model was the strongest predictive model. This suggests that either the selected feature representation is already captured well by a mostly linear decision boundary, or that the tiny positive class makes the tree-based models too conservative or too unstable to learn useful rare-event structure.

Another unexpected result was that logistic regression achieved relatively high recall at the default threshold of 0.5 (around 0.78 on average), which was better than expected given the severity of the class imbalance. However, this came with extremely poor precision (about 0.02), meaning that the model was recovering positives only by flagging a very large number of negatives.

Because logistic regression was the best predictive model, we then used its out-of-fold predicted probabilities to study the decision-threshold tradeoff more directly. We asked whether the model could be tuned to achieve useful recall while meeting a minimum precision requirement.

Under a precision floor of 0.10, the best threshold was about 0.929, which yielded precision = 0.10 and recall ≈ 0.081. In confusion-matrix terms, this meant only 3 of 37 positives were recovered while 27 negatives were incorrectly flagged. This showed that the model does contain real ranking signal, but that the practical precision-recall tradeoff is still quite limited.

We extended this analysis to multiple precision floors (0.05, 0.10, 0.20). As expected, raising the precision requirement increased the threshold and reduced recall. The notable finding here was that moving from 0.10 to 0.20 precision barely changed recall at all: recall remained around 0.081, suggesting that only a very small subset of positives receive truly high-confidence scores.

Overall, the predictive analysis suggests that the finalized engineered features do support meaningful classification signal, but that this signal is best exploited by the simplest model tested, logistic regression. The main limitation is not the complete absence of predictive structure, but rather the severe rarity and overlap of the positive class, which make it difficult to achieve both good recall and acceptable precision at the same time.

## Main Part 5 takeaway

Logistic regression was the strongest predictive model among the three tested approaches, outperforming both nonlinear ensemble alternatives. This was somewhat unexpected, since more flexible tree-based models might have been expected to capture additional structure. However, even the best model showed a harsh precision-recall tradeoff: it could identify some positives, but only a very small subset could be recovered at modest precision levels. In other words, the dataset contains real predictive signal, but the rarity and heterogeneity of exoplanet-labeled stars sharply limit practical detection performance.

In [63]:
from sklearn.linear_model import LogisticRegression
# - compute_threshold_metrics

def evaluate_logistic_cv(
    X,
    y,
    cv_splits,
    C=1.0,
    random_state=42,
    threshold=0.5
):
    """
    Evaluate balanced L2 logistic regression across fixed CV folds.
    Returns per-fold metrics and out-of-fold probabilities.
    """
    fold_rows = []
    oof_prob = np.zeros(len(y), dtype=float)

    for fold_idx, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        clf = LogisticRegression(
            penalty="l2",
            C=C,
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            random_state=random_state
        )

        clf.fit(X_tr, y_tr)
        y_prob = clf.predict_proba(X_val)[:, 1]
        oof_prob[val_idx] = y_prob

        prob_metrics = compute_prob_metrics(y_val, y_prob)
        thr_metrics = compute_threshold_metrics(y_val, y_prob, threshold=threshold)

        row = {
            "fold": fold_idx,
            "n_val": len(val_idx),
            "positives_val": int(y_val.sum()),
            "pr_auc": prob_metrics["pr_auc"],
            "roc_auc": prob_metrics["roc_auc"],
            "precision@0.5": thr_metrics["precision"],
            "recall@0.5": thr_metrics["recall"],
            "f1@0.5": thr_metrics["f1"],
        }
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)

    summary = pd.DataFrame({
        "metric": ["pr_auc", "roc_auc", "precision@0.5", "recall@0.5", "f1@0.5"],
        "mean": [
            fold_df["pr_auc"].mean(),
            fold_df["roc_auc"].mean(),
            fold_df["precision@0.5"].mean(),
            fold_df["recall@0.5"].mean(),
            fold_df["f1@0.5"].mean(),
        ],
        "std": [
            fold_df["pr_auc"].std(ddof=1),
            fold_df["roc_auc"].std(ddof=1),
            fold_df["precision@0.5"].std(ddof=1),
            fold_df["recall@0.5"].std(ddof=1),
            fold_df["f1@0.5"].std(ddof=1),
        ]
    })

    return fold_df, summary, oof_prob


# Run Step 5.1
logit_fold_df, logit_summary, logit_oof_prob = evaluate_logistic_cv(
    X=X_train_scaled,
    y=y_train,
    cv_splits=cv_splits,
    C=1.0,
    threshold=0.5
)

print("Logistic regression CV fold metrics:")
print(logit_fold_df.round(6).to_string(index=False))
print()

print("Logistic regression CV summary:")
print(logit_summary.round(6).to_string(index=False))

Logistic regression CV fold metrics:
 fold  n_val  positives_val   pr_auc  roc_auc  precision@0.5  recall@0.5   f1@0.5
    1   1018              8 0.032709 0.786139       0.021352    0.750000 0.041522
    2   1018              8 0.284509 0.848762       0.024390    0.875000 0.047458
    3   1017              7 0.020854 0.723055       0.017422    0.714286 0.034014
    4   1017              7 0.170221 0.880481       0.022801    1.000000 0.044586
    5   1017              7 0.019900 0.743706       0.013333    0.571429 0.026059

Logistic regression CV summary:
       metric     mean      std
       pr_auc 0.105639 0.118347
      roc_auc 0.796429 0.067218
precision@0.5 0.019860 0.004471
   recall@0.5 0.782143 0.162882
       f1@0.5 0.038728 0.008676


c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated

In [64]:
from sklearn.ensemble import RandomForestClassifier

def evaluate_random_forest_cv(
    X,
    y,
    cv_splits,
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    threshold=0.5
):
    """
    Evaluate Random Forest across fixed CV folds.
    Returns per-fold metrics and out-of-fold probabilities.
    """
    fold_rows = []
    oof_prob = np.zeros(len(y), dtype=float)

    for fold_idx, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced_subsample",
            random_state=random_state,
            n_jobs=-1
        )

        clf.fit(X_tr, y_tr)
        y_prob = clf.predict_proba(X_val)[:, 1]
        oof_prob[val_idx] = y_prob

        prob_metrics = compute_prob_metrics(y_val, y_prob)
        thr_metrics = compute_threshold_metrics(y_val, y_prob, threshold=threshold)

        row = {
            "fold": fold_idx,
            "n_val": len(val_idx),
            "positives_val": int(y_val.sum()),
            "pr_auc": prob_metrics["pr_auc"],
            "roc_auc": prob_metrics["roc_auc"],
            "precision@0.5": thr_metrics["precision"],
            "recall@0.5": thr_metrics["recall"],
            "f1@0.5": thr_metrics["f1"],
        }
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)

    summary = pd.DataFrame({
        "metric": ["pr_auc", "roc_auc", "precision@0.5", "recall@0.5", "f1@0.5"],
        "mean": [
            fold_df["pr_auc"].mean(),
            fold_df["roc_auc"].mean(),
            fold_df["precision@0.5"].mean(),
            fold_df["recall@0.5"].mean(),
            fold_df["f1@0.5"].mean(),
        ],
        "std": [
            fold_df["pr_auc"].std(ddof=1),
            fold_df["roc_auc"].std(ddof=1),
            fold_df["precision@0.5"].std(ddof=1),
            fold_df["recall@0.5"].std(ddof=1),
            fold_df["f1@0.5"].std(ddof=1),
        ]
    })

    return fold_df, summary, oof_prob


# Run Step 5.2
rf_fold_df, rf_summary, rf_oof_prob = evaluate_random_forest_cv(
    X=X_train_unscaled,
    y=y_train,
    cv_splits=cv_splits,
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    threshold=0.5
)

print("Random Forest CV fold metrics:")
print(rf_fold_df.round(6).to_string(index=False))
print()

print("Random Forest CV summary:")
print(rf_summary.round(6).to_string(index=False))

Random Forest CV fold metrics:
 fold  n_val  positives_val   pr_auc  roc_auc  precision@0.5  recall@0.5  f1@0.5
    1   1018              8 0.011954 0.600804            0.0         0.0     0.0
    2   1018              8 0.012023 0.635025            0.0         0.0     0.0
    3   1017              7 0.019263 0.685926            0.0         0.0     0.0
    4   1017              7 0.080187 0.597878            0.0         0.0     0.0
    5   1017              7 0.023370 0.713366            0.0         0.0     0.0

Random Forest CV summary:
       metric     mean      std
       pr_auc 0.029359 0.028831
      roc_auc 0.646600 0.051501
precision@0.5 0.000000 0.000000
   recall@0.5 0.000000 0.000000
       f1@0.5 0.000000 0.000000


In [65]:
from sklearn.ensemble import GradientBoostingClassifier

def evaluate_gradient_boosting_cv(
    X,
    y,
    cv_splits,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    subsample=1.0,
    random_state=42,
    threshold=0.5
):
    """
    Evaluate Gradient Boosting across fixed CV folds.
    Returns per-fold metrics and out-of-fold probabilities.
    """
    fold_rows = []
    oof_prob = np.zeros(len(y), dtype=float)

    for fold_idx, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        clf = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            random_state=random_state
        )

        clf.fit(X_tr, y_tr)
        y_prob = clf.predict_proba(X_val)[:, 1]
        oof_prob[val_idx] = y_prob

        prob_metrics = compute_prob_metrics(y_val, y_prob)
        thr_metrics = compute_threshold_metrics(y_val, y_prob, threshold=threshold)

        row = {
            "fold": fold_idx,
            "n_val": len(val_idx),
            "positives_val": int(y_val.sum()),
            "pr_auc": prob_metrics["pr_auc"],
            "roc_auc": prob_metrics["roc_auc"],
            "precision@0.5": thr_metrics["precision"],
            "recall@0.5": thr_metrics["recall"],
            "f1@0.5": thr_metrics["f1"],
        }
        fold_rows.append(row)

    fold_df = pd.DataFrame(fold_rows)

    summary = pd.DataFrame({
        "metric": ["pr_auc", "roc_auc", "precision@0.5", "recall@0.5", "f1@0.5"],
        "mean": [
            fold_df["pr_auc"].mean(),
            fold_df["roc_auc"].mean(),
            fold_df["precision@0.5"].mean(),
            fold_df["recall@0.5"].mean(),
            fold_df["f1@0.5"].mean(),
        ],
        "std": [
            fold_df["pr_auc"].std(ddof=1),
            fold_df["roc_auc"].std(ddof=1),
            fold_df["precision@0.5"].std(ddof=1),
            fold_df["recall@0.5"].std(ddof=1),
            fold_df["f1@0.5"].std(ddof=1),
        ]
    })

    return fold_df, summary, oof_prob


gb_fold_df, gb_summary, gb_oof_prob = evaluate_gradient_boosting_cv(
    X=X_train_unscaled,
    y=y_train,
    cv_splits=cv_splits,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    subsample=1.0,
    threshold=0.5
)

print("Gradient Boosting CV fold metrics:")
print(gb_fold_df.round(6).to_string(index=False))
print()

print("Gradient Boosting CV summary:")
print(gb_summary.round(6).to_string(index=False))

Gradient Boosting CV fold metrics:
 fold  n_val  positives_val   pr_auc  roc_auc  precision@0.5  recall@0.5  f1@0.5
    1   1018              8 0.014747 0.700990            0.0         0.0     0.0
    2   1018              8 0.025491 0.819926            0.0         0.0     0.0
    3   1017              7 0.020075 0.766337            0.0         0.0     0.0
    4   1017              7 0.030373 0.775460            0.0         0.0     0.0
    5   1017              7 0.016076 0.722560            0.0         0.0     0.0

Gradient Boosting CV summary:
       metric     mean      std
       pr_auc 0.021353 0.006550
      roc_auc 0.757054 0.046666
precision@0.5 0.000000 0.000000
   recall@0.5 0.000000 0.000000
       f1@0.5 0.000000 0.000000


In [66]:
def evaluate_threshold_grid(y_true, y_prob, thresholds=None):
    """
    Evaluate precision/recall/F1 over a grid of thresholds.
    """
    if thresholds is None:
        thresholds = np.linspace(0.0, 1.0, 1001)

    rows = []
    for t in thresholds:
        metrics = compute_threshold_metrics(y_true, y_prob, threshold=t)
        rows.append({
            "threshold": t,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        })

    return pd.DataFrame(rows)


# --------------------------------------------------
# Main Q2 threshold analysis
# --------------------------------------------------

min_precision = 0.10

threshold_df = evaluate_threshold_grid(y_train, logit_oof_prob)

best_threshold_result = find_threshold_for_min_precision(
    y_true=y_train,
    y_prob=logit_oof_prob,
    min_precision=min_precision,
    n_steps=1001
)

print("Best threshold under precision constraint:")
print(best_threshold_result)
print()

# Show a few candidate thresholds meeting the precision floor
eligible = threshold_df[threshold_df["precision"] >= min_precision].copy()
eligible = eligible.sort_values(["recall", "f1"], ascending=[False, False])

print("Top candidate thresholds with precision >= {:.2f}:".format(min_precision))
print(eligible.head(10).round(6).to_string(index=False))

Best threshold under precision constraint:
{'threshold': np.float64(0.929), 'precision': 0.1, 'recall': 0.08108108108108109, 'f1': 0.08955223880597014}

Top candidate thresholds with precision >= 0.10:
 threshold  precision   recall       f1
     0.949   0.214286 0.081081 0.117647
     0.950   0.214286 0.081081 0.117647
     0.951   0.214286 0.081081 0.117647
     0.952   0.214286 0.081081 0.117647
     0.946   0.200000 0.081081 0.115385
     0.947   0.200000 0.081081 0.115385
     0.948   0.200000 0.081081 0.115385
     0.945   0.187500 0.081081 0.113208
     0.944   0.176471 0.081081 0.111111
     0.942   0.166667 0.081081 0.109091


In [67]:
def plot_threshold_tradeoff(threshold_df, min_precision=0.10,
                            out_path="./plots/logistic_threshold_tradeoff.png"):
    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
    ax.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
    ax.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")

    ax.axhline(min_precision, linestyle="--", linewidth=1, label=f"Precision floor = {min_precision:.2f}")

    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Metric value")
    ax.set_title("Logistic regression threshold tradeoff (OOF predictions)")
    ax.legend()

    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()

plot_threshold_tradeoff(threshold_df, min_precision=min_precision)

In [68]:
chosen_threshold = best_threshold_result["threshold"]

chosen_metrics = compute_threshold_metrics(
    y_true=y_train,
    y_prob=logit_oof_prob,
    threshold=chosen_threshold
)

print("Chosen threshold:", round(chosen_threshold, 6))
print("Precision:", round(chosen_metrics["precision"], 6))
print("Recall   :", round(chosen_metrics["recall"], 6))
print("F1       :", round(chosen_metrics["f1"], 6))
print("Confusion matrix:")
print(chosen_metrics["confusion_matrix"])

Chosen threshold: 0.929
Precision: 0.1
Recall   : 0.081081
F1       : 0.089552
Confusion matrix:
[[5023   27]
 [  34    3]]


In [69]:
# Run the same threshold search for precision floors 0.05, 0.10, and 0.20 to see which ones are stronger, and show a tradeoff curve

precision_floors = [0.05, 0.10, 0.20]
threshold_results = []

for pf in precision_floors:
    result = find_threshold_for_min_precision(
        y_true=y_train,
        y_prob=logit_oof_prob,
        min_precision=pf,
        n_steps=1001
    )
    result["precision_floor"] = pf
    threshold_results.append(result)

threshold_results_df = pd.DataFrame(threshold_results)
print("Threshold search results for multiple precision floors:")
print(threshold_results_df.round(6).to_string(index=False))

Threshold search results for multiple precision floors:
 threshold  precision   recall       f1  precision_floor
     0.867       0.05 0.135135 0.072993             0.05
     0.929       0.10 0.081081 0.089552             0.10
     0.946       0.20 0.081081 0.115385             0.20


In [70]:
import pandas as pd

def get_metric(summary_df, metric_name):
    return summary_df.loc[summary_df["metric"] == metric_name, "mean"].iloc[0]

def get_metric_std(summary_df, metric_name):
    return summary_df.loc[summary_df["metric"] == metric_name, "std"].iloc[0]

model_comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosting"],
    "PR-AUC Mean": [
        get_metric(logit_summary, "pr_auc"),
        get_metric(rf_summary, "pr_auc"),
        get_metric(gb_summary, "pr_auc"),
    ],
    "PR-AUC Std": [
        get_metric_std(logit_summary, "pr_auc"),
        get_metric_std(rf_summary, "pr_auc"),
        get_metric_std(gb_summary, "pr_auc"),
    ],
    "ROC-AUC Mean": [
        get_metric(logit_summary, "roc_auc"),
        get_metric(rf_summary, "roc_auc"),
        get_metric(gb_summary, "roc_auc"),
    ],
    "Precision@0.5": [
        get_metric(logit_summary, "precision@0.5"),
        get_metric(rf_summary, "precision@0.5"),
        get_metric(gb_summary, "precision@0.5"),
    ],
    "Recall@0.5": [
        get_metric(logit_summary, "recall@0.5"),
        get_metric(rf_summary, "recall@0.5"),
        get_metric(gb_summary, "recall@0.5"),
    ],
    "F1@0.5": [
        get_metric(logit_summary, "f1@0.5"),
        get_metric(rf_summary, "f1@0.5"),
        get_metric(gb_summary, "f1@0.5"),
    ],
})

model_comparison_df.round(4)

,Model,PR-AUC Mean,PR-AUC Std,ROC-AUC Mean,Precision@0.5,Recall@0.5,F1@0.5
0,Logistic Regression,0.1056,0.1183,0.7964,0.0199,0.7821,0.0387
1,Random Forest,0.0294,0.0288,0.6466,0.0000,0.0000,0.0000
2,Gradient Boosting,0.0214,0.0066,0.7571,0.0000,0.0000,0.0000


In [71]:
os.makedirs("tables", exist_ok=True)
inferential_summary_labeled.to_csv("tables/inferential_summary.csv", index=False)
model_comparison_df.to_csv("tables/model_comparison.csv", index=False)
threshold_results_df.to_csv("tables/threshold_results.csv", index=False)